In [143]:
import pandas as pd
import re
import nltk
import string
from utils import *
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from openai import OpenAI
import openai
import fitz
from openai import OpenAI
import openai

## Stopwords Preprocess

In [144]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nowegian_stopwords = stopwords.words('norwegian')
stopwords = []
for x in nowegian_stopwords:
  if x =='ja':
    pass
  else:
    stopwords.append(x)

def puntuation_preprocess(text):
  p = re.compile("[" + re.escape(string.punctuation) + "]")
  return re.sub(' +',' ',p.sub("", text).lower())

def remove_stop_words(text):

    text_tokens = word_tokenize(text)
    text_tokens = [word for word in text_tokens if not word in stopwords]

    return ' '.join(text_tokens)

def preprocess(text):
  text = remove_stop_words(puntuation_preprocess(text))
  return text

In [145]:
##Openai 

In [146]:
client = OpenAI(api_key ='sk-proj-Qjb8HmX7anyLgGATenZaT3BlbkFJ2rxn0QXqk1ldzQmjjSiG')



In [147]:
def gpt_embedding(text):
  response = client.embeddings.create(
    input=text,
    model="text-embedding-ada-002"
  )

  return response.data[0].embedding




In [148]:
user_questions = [
        'Kjenner du til om det er utført arbeid på bad/våtrom?',
        "ble det lagt ny membran?",
        "Foreligger det samsvarserklæring?",
        "er det foretatt radon måling?",
        "er det automat eller skrusikringer?"
    ]

In [149]:
##loading the values from the pdf and questions

## Questions Detection from pdfs and to pandas df

In [150]:
checkbox_questions = []



In [163]:

def pymypdf_reader(filename):

    doc = fitz.open(filename)
    text = ""
    for page in doc:
        text  =  text +page.get_text()
    return text

def read_the_pdf_and_checkbox_text(filename):
  text = pymypdf_reader(filename)
  lines = text.splitlines()
  a_count = 0
  j = 0

  checkbox_state = False
  information_text = []

  for i in range(len(lines)):
    if lines[i] == "Dette skjema vil være en del av salgsoppgaven" or  lines[i] == "Til orientering vil dette skjema være en del av salgsoppgaven":
        checkbox_state = True
    if checkbox_state == True:
        ##information
        if lines[i] =='Meglerfirma':
            information_text.append(
            [lines[i],lines[i+1]]
            )
        elif lines[i] =='Oppdragsnr':
            information_text.append(
            [lines[i],lines[i+1]]
            )
        elif lines[i] =='Selger 1 navn':
            information_text.append(
            [lines[i],lines[i+1]]
            )
        elif lines[i] =='I hvilket forsikringsselskap har du tegnet villa/husforsikring?':

            information_text.append(
            [
                lines[i],lines[i+1] + ' ' + lines[i+2] + ' ' + lines[i+3]  #+ ' ' + lines[1+4]  not able to detect the policy number
            ]
            )

        elif lines[i]=='Hjemmelshavers navn':
            information_text.append(
               [ lines[i] , lines[i+1]    ]
            )
        elif lines[i] == 'Hvor lenge har du eid boligen?':
            information_text.append(
               [ lines[i] , lines[i+1] +' '+lines[i+2]+' '+lines[i+3]+' '+lines[i+4]    ]
            )

        elif lines[i] =='Når kjøpte du boligen?':
            information_text.append(
                [lines[i], lines[i+1] +' '+lines[i+2] ]
            )
        #Beskrivelse
        elif lines[i] =='Kjenner du til om det er/har vært utført arbeid på el-anlegget eller andre installasjoner (f.eks. oljetank, sentralfyr, ventilasjon)?':
            information_text.append(
                [lines[i],  lines[i+1] + ' ' + lines[i+2]  + ' ' + lines[i+3] + ' ' + lines[i+4] + ' ' + lines[i+5] + ' ' + lines[i+6] + ' '  ]
            )

        elif lines[i] =='Kjenner du til om det er utført arbeid på bad/våtrom?':

            information_text.append(
                [lines[i], lines[i+1] +' '+lines[i+2] ]
            )
        elif lines[i] =='Kjenner du til feil eller om har vært utført arbeid/kontroll på vann/avløp?':

            information_text.append(
                [lines[i], lines[i+1] +' '+lines[i+2] ]
            )
        elif lines[i] =='Kjenner du til om det har vært utført arbeid på terrasse/garasje/tak/fasade?':

            information_text.append(
                [lines[i], lines[i+1] +' '+lines[i+2] ]
            )
        else:
            pass

  return text,information_text


def preprocess_with_namespace_token(text):
  text = text.strip()
  return text.replace('\n',' ')

def checkbox_detector(filename):
    checked_values = []
    pdf = fitz.open(filename)
    checked_values = []
    for page in pdf:
            paths = page.get_drawings()
            text = page.get_text()
            words = [w for w in page.get_text(
                "words") if w[4] in ("Nei", "Ja")]

            for w in words:
                r = fitz.Rect(w[:4]) + (-2, -2, 2, 2)  # rect a bit larger
                r.x0 -= 2*r.width  # enlarge to the left to catch checkbox
                l = len([p for p in paths if p["rect"] in r])

                if l >= 6:
                    checked_values.append((w[4]))

    return checked_values

def checkbox_values(filename):
    checkbox_questions_answere_text = []
    
    checkbox_questions=[
        'Er det dødsbo?',
        'Er det salg ved fullmakt?',
        'Har du kjennskap til eiendommen?',
        'Har du bodd i boligen siste 12 måneder?',
        'Kjenner du til om det er/har vært feil tilknyttet våtrommene, f.eks. sprekker, lekkasje, råte, lukt eller soppskader?',
        'Ble tettesjikt/membran/sluk oppgradert/fornyet?',
        'Er arbeidet byggemeldt?',
        'Kjenner du til om det er/har vært tilbakeslag av avløpsvann i sluk eller lignende?',
        'Kjenner du til feil eller om har vært utført arbeid/kontroll på vann/avløp?',
        'Kjenner du til om det er/har vært utettheter i terrasse/garasje/tak/fasade?',
        'Kjenner du til om det er/har vært problemer med ildsted/skorstein/pipe f.eks. dårlig trekk, sprekker, pålegg, fyringsforbud eller lignende?',
        'Kjenner du til om det er/har vært f.eks. sprekker i mur, skjeve gulv eller lignende?',
        'Kjenner du til om det er/har vært sopp/råteskader/insekter/skadedyr på eiendommen som rotter, mus, maur eller lignende?',
        'Kjenner du til om det er/har vært skjeggkre i boligen?',
        'Foreligger det samsvarserklæring (i henhold til forskrift om elektriske lavspenningsanlegg)?',
        'Kjenner du til om det er utført kontroll av el-anlegget og/eller andre installasjoner (f.eks. oljetank, sentralfyr, ventilasjon)?',
        'Har du ladeanlegg/ladeboks for elbil i dag?',
        'Kjenner du til om ufaglærte har utført arbeid som normalt bør utføres av faglærte, utover det som er nevnt tidligere (f.eks. drenering,murerarbeid, tømrerarbeid etc)?',
        'Er det nedgravd oljetank på eiendommen?',
        'Selges eiendommen med utleiedel, leilighet, hybel eller tilsvarende?',
        'Kjenner du til om det er innredet/bruksendret/bygget ut i kjeller eller loft eller andre deler av boligen?',
        'Kjenner du til forslag eller vedtatte reguleringsplaner, andre planer, nabovarsel eller offentlige vedtak som kan medføre endringer i bruken av eiendommen eller av dens omgivelser?',
        'Kjenner du til om det foreligger påbud/heftelser/krav/manglende tillatelser vedrørende eiendommen?',
        'Er det foretatt radonmåling?',
        'Kjenner du til manglende brukstillatelse eller ferdigattest?',
        'Kjenner du til om det foreligger skaderapporter/ tilstandsvurderinger eller utførte målinger?',
        'Er det andre forhold av betydning ved eiendommen som kan være relevant for kjøper å vite om (f.eks. rasfare, tinglyste forhold eller private avtaler)?',
        'Kjenner du til om sameiet/laget/selskapet er involvert i tvister av noe slag?',
        'Kjenner du til vedtak/forslag til vedtak om forhold vedr. eiendommen som kan medføre økte felleskostnader/økt fellesgjeld?',
        'Kjenner du til om det er/har vært sopp/råteskader/insekter/skadedyr i sameiet/laget/selskapet (fellesareal eller i andre boliger) som rotter, mus, maur eller lignende?',
        'Kjenner du til om det er/har vært skjeggkre i sameiet/laget/selskapet (fellesareal eller i andre boliger)?',
    ]

    ##getting the checkboxvalue
    checkbox_values = checkbox_detector(filename)

    ##combining the checkbox values
    text,information_text = read_the_pdf_and_checkbox_text(filename)

    for i,_ in enumerate(checkbox_values):
        checkbox_questions_answere_text.append([preprocess(checkbox_questions[i]),checkbox_values[i]])
   
    #for i,_ in enumerate(information_text):
    #    checkbox_questions_answere_text.append([checkbox_questions[i] ,checkbox_values[i]])
    
    return checkbox_questions_answere_text



### Generating the questions embeddings

In [164]:
def generating_pdf_questions_embedding(pdf_questions):
    question_embeddings = []
    vector_dict = []
    for x in pdf_questions:
      vector_dict.append([
      gpt_embedding(preprocess(x)),
      preprocess(x),
      ]
      )
    return vector_dict

In [165]:
question_answeres_checkboxs = checkbox_values('data/output1.pdf')

In [166]:
df = pd.DataFrame(question_answeres_checkboxs,columns=['questions','answere'])

In [167]:
questions = df['questions'].values.tolist()

In [168]:
#questions

In [169]:
questions_embedding = generating_pdf_questions_embedding(questions)

In [170]:
user_questions = [
        'Kjenner du til om det er utført arbeid på bad/våtrom?',
        "ble det lagt ny membran?",
        "Foreligger det samsvarserklæring?",
        "er det foretatt radon måling?",
        "er det automat eller skrusikringer?"
    ]

In [171]:
def predict(question_ask):
  ##implementing the preprocess

  vector = gpt_embedding(preprocess(question_ask))
  scores = []
  for i,x in enumerate(questions_embedding):
    scores.append([
        cosine_similarity(np.array(vector).reshape(1,-1),
        np.array(x[0]).reshape(1,-1)),x[-1]
    ])

  vector_df = pd.DataFrame(scores,columns=['score','index'])
  question = vector_df.sort_values(by=['score'],ascending=False).to_numpy().tolist()[0][-1]


    
  answere = df[df['questions']==question.strip()]['answere'].values.tolist()[0]

    
  return question,answere

In [172]:
df['questions']

0                                                dødsbo
1                                         salg fullmakt
2                                  kjennskap eiendommen
3                         bodd boligen siste 12 måneder
4     kjenner erhar feil tilknyttet våtrommene feks ...
5               tettesjiktmembransluk oppgradertfornyet
6                                   arbeidet byggemeldt
7     kjenner erhar tilbakeslag avløpsvann sluk lign...
8          kjenner feil utført arbeidkontroll vannavløp
9     kjenner erhar utettheter terrassegarasjetakfasade
10    kjenner erhar problemer ildstedskorsteinpipe f...
11    kjenner erhar feks sprekker mur skjeve gulv li...
12    kjenner erhar soppråteskaderinsekterskadedyr e...
13                      kjenner erhar skjeggkre boligen
14    foreligger samsvarserklæring henhold forskrift...
15    kjenner utført kontroll elanlegget ogeller and...
16                         ladeanleggladeboks elbil dag
17    kjenner ufaglærte utført arbeid normalt bø

In [173]:
#df[df['questions'] =='Er det dødsbo?']['answere'].values.tolist()[0]

In [174]:
for x in user_questions:
    question,answere = predict(x)
    print(x)
    print("*"*20)
    print("answere",answere)
    print("*"*40)

Kjenner du til om det er utført arbeid på bad/våtrom?
********************
answere Nei
****************************************
ble det lagt ny membran?
********************
answere Nei
****************************************
Foreligger det samsvarserklæring?
********************
answere Ja
****************************************
er det foretatt radon måling?
********************
answere Nei
****************************************
er det automat eller skrusikringer?
********************
answere Nei
****************************************


## Applying the preprocess

In [137]:
##apply preprocess

In [ ]:
predict()

In [86]:
#questions_embedding

In [82]:
#question_answeres_checkboxs

In [ ]:
question_answeres_checkboxs[]

In [17]:
def predict(question_ask):
  ##implementing the preprocess
  question_ask = preprocess(question_ask)

  print(question_ask)

  vector = gpt_embedding(question_ask)
  scores = []
  for i,x in enumerate(vector_dict):
    scores.append([
        cosine_similarity(np.array(vector).reshape(1,-1),
        np.array(x[0]).reshape(1,-1)),x[-1]
    ])
    #scores.append([cosine_similarity(np.asarray(vector).reshape(-1,1),
    #                np.asarray(x[0]).reshape(-1,1) ),x[-1]])

  vector_df = pd.DataFrame(scores,columns=['score','index'])
  question = vector_df.sort_values(by=['score'],ascending=False).to_numpy().tolist()[0][-1]
  dff = vector_df.sort_values(by=['score'],ascending=False).to_numpy().tolist()[0]
  return question,dff